In [ ]:
import os 
os.chdir("/hpc/home/ephdh/workspace/suzhou_false_validation/analysis")

import ast
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint

In [ ]:
result_df = pd.read_csv(
        "/hpc/home/ephdh/workspace/suzhou_false_validation/data/meta_data/meta_data_with_model_scores.csv", 
        encoding="utf-16",  
        converters={"DICOMPaths": ast.literal_eval, 
                    'clip_image_scores': ast.literal_eval,
                    'cnn_image_scores': ast.literal_eval}
        )
result_df['DICOM_paths'] = result_df['DICOM_paths'].apply(ast.literal_eval)

In [ ]:
with pd.option_context('display.max_columns', None):
    display(result_df.head())

In [ ]:
# ==========================================
# 2. BOOTSTRAPPING LOGIC
# ==========================================

def calculate_metrics_single_batch(y_true, y_pred, y_prob):
    """Calculates metrics for a single bootstrap sample"""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Handle potential division by zero in small/imbalanced samples
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0 # Handle edge case where sample has only 1 class
        
    return sens, spec, auc

def bootstrap_all_metrics(y_true, y_pred, y_prob, n_bootstraps=2000):
    """
    Resamples the dataset N times and calculates the 95% CI 
    (2.5th and 97.5th percentiles) for each metric.
    """
    rng = np.random.RandomState(42)
    sens_boot, spec_boot, auc_boot = [], [], []
    n_samples = len(y_true)
    
    print(f"Starting Bootstrapping ({n_bootstraps} iterations)...")
    
    for i in range(n_bootstraps):
        # 1. Resample indices with replacement
        indices = rng.randint(0, n_samples, n_samples)
        
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]
        y_prob_boot = y_prob[indices]
        
        # 2. Skip samples that don't have both classes (required for AUC)
        if len(np.unique(y_true_boot)) < 2:
            continue
            
        # 3. Calculate metrics for this sample
        sens, spec, auc = calculate_metrics_single_batch(y_true_boot, y_pred_boot, y_prob_boot)
        
        sens_boot.append(sens)
        spec_boot.append(spec)
        auc_boot.append(auc)
        
    # 4. Helper to get Percentiles
    def get_ci_bounds(data):
        sorted_data = np.sort(np.array(data))
        lower = sorted_data[int(0.025 * len(sorted_data))]
        upper = sorted_data[int(0.975 * len(sorted_data))]
        mean_val = np.mean(sorted_data)
        return mean_val, lower, upper

    return {
        'sens': get_ci_bounds(sens_boot),
        'spec': get_ci_bounds(spec_boot),
        'auc': get_ci_bounds(auc_boot)
    }

def adjust_ppv_npv(sens, spec, prev):
    """Calculates Adjusted PPV/NPV using Bayes' Theorem"""
    ppv = (sens * prev) / ((sens * prev) + ((1 - spec) * (1 - prev)))
    npv = (spec * (1 - prev)) / ((spec * (1 - prev)) + ((1 - sens) * prev))
    return ppv, npv

In [ ]:
df = result_df.copy()

# Prepare Arrays
y_true = df['cancer'].values
y_pred = df['ai_pos'].values
y_prob = df['ensemble_score'].values

# Run Bootstrap
results = bootstrap_all_metrics(y_true, y_pred, y_prob, n_bootstraps=2000)

# Calculate Prevalence Adjustments (using the mean bootstrap values)
prev_screen = 0.005
mean_sens = results['sens'][0]
mean_spec = results['spec'][0]
ppv_adj, npv_adj = adjust_ppv_npv(mean_sens, mean_spec, prev_screen)

print("\n" + "="*65)
print("TABLE 2: GLOBAL DIAGNOSTIC PERFORMANCE (With Bootstrap CIs)")
print("="*65)
print(f"{'Metric':<25} | {'Value':<10} | {'95% CI (Bootstrap)':<20}")
print("-" * 65)

# AUC
val, low, high = results['auc']
print(f"{'AUC-ROC':<25} | {val:.3f}      | ({low:.3f} – {high:.3f})")

# Sensitivity
val, low, high = results['sens']
print(f"{'Sensitivity (Recall)':<25} | {val:.1%}      | ({low:.1%} – {high:.1%})")

# Specificity
val, low, high = results['spec']
print(f"{'Specificity':<25} | {val:.1%}      | ({low:.1%} – {high:.1%})")

print("-" * 65)
print(f"Prevalence-Adjusted Metrics (Assumed Prev={prev_screen:.1%})")
print(f"{'Adj. PPV':<25} | {ppv_adj:.2%}      | N/A (Derived)")
print(f"{'Adj. NPV':<25} | {npv_adj:.2%}      | N/A (Derived)")
print("="*65)

In [ ]:
# ==========================================
# 1. LOAD DATA
# ==========================================
# REPLACE with your actual file loading:
# df = pd.read_csv('your_dataset.csv')

# --- MOCK DATA GENERATOR (Matching your N=900 dataset) ---
np.random.seed(42)
data = []
# Stratum 1: TP (Concordant Cancer) - n=250
data.extend([[1, 1, np.random.uniform(0.7, 0.99)] for _ in range(250)])
# Stratum 2: FN (Discordant Cancer/Rescue) - n=200
data.extend([[1, 0, np.random.uniform(0.01, 0.45)] for _ in range(150)]) # Missed
data.extend([[1, 1, np.random.uniform(0.55, 0.85)] for _ in range(50)])  # Rescued
# Stratum 1: TN (Concordant Benign) - n=250
data.extend([[0, 0, np.random.uniform(0.01, 0.3)] for _ in range(250)])
# Stratum 2: FP (Discordant Benign/Avoidance) - n=200
data.extend([[0, 1, np.random.uniform(0.51, 0.8)] for _ in range(100)]) # Failed
data.extend([[0, 0, np.random.uniform(0.2, 0.49)] for _ in range(100)]) # Avoided

df = pd.DataFrame(data, columns=['cancer', 'ai_pos', 'ensemble_score'])
# ---------------------------------------------------------

# ==========================================
# 2. BOOTSTRAPPING LOGIC
# ==========================================

def calculate_metrics_single_batch(y_true, y_pred, y_prob):
    """Calculates metrics for a single bootstrap sample"""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Handle potential division by zero in small/imbalanced samples
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0 # Handle edge case where sample has only 1 class
        
    return sens, spec, auc

def bootstrap_all_metrics(y_true, y_pred, y_prob, n_bootstraps=2000):
    """
    Resamples the dataset N times and calculates the 95% CI 
    (2.5th and 97.5th percentiles) for each metric.
    """
    rng = np.random.RandomState(42)
    sens_boot, spec_boot, auc_boot = [], [], []
    n_samples = len(y_true)
    
    print(f"Starting Bootstrapping ({n_bootstraps} iterations)...")
    
    for i in range(n_bootstraps):
        # 1. Resample indices with replacement
        indices = rng.randint(0, n_samples, n_samples)
        
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]
        y_prob_boot = y_prob[indices]
        
        # 2. Skip samples that don't have both classes (required for AUC)
        if len(np.unique(y_true_boot)) < 2:
            continue
            
        # 3. Calculate metrics for this sample
        sens, spec, auc = calculate_metrics_single_batch(y_true_boot, y_pred_boot, y_prob_boot)
        
        sens_boot.append(sens)
        spec_boot.append(spec)
        auc_boot.append(auc)
        
    # 4. Helper to get Percentiles
    def get_ci_bounds(data):
        sorted_data = np.sort(np.array(data))
        lower = sorted_data[int(0.025 * len(sorted_data))]
        upper = sorted_data[int(0.975 * len(sorted_data))]
        mean_val = np.mean(sorted_data)
        return mean_val, lower, upper

    return {
        'sens': get_ci_bounds(sens_boot),
        'spec': get_ci_bounds(spec_boot),
        'auc': get_ci_bounds(auc_boot)
    }

def adjust_ppv_npv(sens, spec, prev):
    """Calculates Adjusted PPV/NPV using Bayes' Theorem"""
    ppv = (sens * prev) / ((sens * prev) + ((1 - spec) * (1 - prev)))
    npv = (spec * (1 - prev)) / ((spec * (1 - prev)) + ((1 - sens) * prev))
    return ppv, npv

# ==========================================
# 3. EXECUTE & PRINT
# ==========================================

# Prepare Arrays
y_true = df['cancer'].values
y_pred = df['ai_pos'].values
y_prob = df['ensemble_score'].values

# Run Bootstrap
results = bootstrap_all_metrics(y_true, y_pred, y_prob, n_bootstraps=2000)

# Calculate Prevalence Adjustments (using the mean bootstrap values)
prev_screen = 0.005
mean_sens = results['sens'][0]
mean_spec = results['spec'][0]
ppv_adj, npv_adj = adjust_ppv_npv(mean_sens, mean_spec, prev_screen)

print("\n" + "="*65)
print("TABLE 2: GLOBAL DIAGNOSTIC PERFORMANCE (With Bootstrap CIs)")
print("="*65)
print(f"{'Metric':<25} | {'Value':<10} | {'95% CI (Bootstrap)':<20}")
print("-" * 65)

# AUC
val, low, high = results['auc']
print(f"{'AUC-ROC':<25} | {val:.3f}      | ({low:.3f} – {high:.3f})")

# Sensitivity
val, low, high = results['sens']
print(f"{'Sensitivity (Recall)':<25} | {val:.1%}      | ({low:.1%} – {high:.1%})")

# Specificity
val, low, high = results['spec']
print(f"{'Specificity':<25} | {val:.1%}      | ({low:.1%} – {high:.1%})")

print("-" * 65)
print(f"Prevalence-Adjusted Metrics (Assumed Prev={prev_screen:.1%})")
print(f"{'Adj. PPV':<25} | {ppv_adj:.2%}      | N/A (Derived)")
print(f"{'Adj. NPV':<25} | {npv_adj:.2%}      | N/A (Derived)")
print("="*65)

In [ ]:

# ==========================================
# 4. STRATIFIED ANALYSIS (Based on your Separation Request)
# ==========================================
print("\n### COHORT-SPECIFIC PERFORMANCE ###")
print(f"{'Cohort':<25} | {'N':<5} | {'Goal':<15} | {'AI Performance':<20}")
print("-" * 75)

# A. Concordant Malignant (Historical TP)
# Goal: Maintain High Sensitivity (Safety)
grp_tp = df[df['cohort_group'] == 'TP_Concordant']
ai_hits_tp = grp_tp['ai_binary'].sum()
acc_tp = ai_hits_tp / len(grp_tp)
print(f"{'Concordant Cancer (TP)':<25} | {len(grp_tp):<5} | {'Safety Check':<15} | {acc_tp:.1%} Detected")

# B. Concordant Benign (Historical TN)
# Goal: Maintain High Specificity (Efficiency)
grp_tn = df[df['cohort_group'] == 'TN_Concordant']
ai_corr_tn = (grp_tn['ai_binary'] == 0).sum()
acc_tn = ai_corr_tn / len(grp_tn)
print(f"{'Concordant Benign (TN)':<25} | {len(grp_tn):<5} | {'Efficiency':<15} | {acc_tn:.1%} Correctly Cleared")

# C. Discordant Malignant (Historical FN)
# Goal: Rescue these missed cancers
grp_fn = df[df['cohort_group'] == 'FN_Discordant']
ai_rescue = grp_fn['ai_binary'].sum()
rescue_rate = ai_rescue / len(grp_fn)
print(f"{'Discordant Cancer (FN)':<25} | {len(grp_fn):<5} | {'Rescue Value':<15} | {rescue_rate:.1%} RESCUED")

# D. Discordant Benign (Historical FP)
# Goal: Avoid these unnecessary biopsies
grp_fp = df[df['cohort_group'] == 'FP_Discordant']
ai_avoid = (grp_fp['ai_binary'] == 0).sum()
avoid_rate = ai_avoid / len(grp_fp)
print(f"{'Discordant Benign (FP)':<25} | {len(grp_fp):<5} | {'Avoidance':<15} | {avoid_rate:.1%} AVOIDED")
print("-" * 75)